## Load and prepare final dataframe for model

In [1]:
# Load df_artists and df_songs from CSV with all columns visible in head/sample output.

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

df_artists = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_artists.csv')

print(df_artists.shape)

# Load df_songs from CSV.

df_songs = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_songs.csv')

print(df_songs.shape)

(13655, 44)
(38383, 43)


In [2]:
# Create df_artists_final from df_artists:
# - drops non-feature/identifier columns
# - filters to artists whose first top-20 hit was between 2000 and 2019
# - keeps combined_major_genre_categories, renamed to combined_major_genre_categories_artist

cols_to_drop = [
    'musicbrainz_artist_id', 'musicbrainz_mbid', 'spotify_id',
    'performer_pre_normalized', 'last_charting_song_year', 'first_song_year',
    'years_active_on_charts', 'first_charting_song_position',
    'first_charting_song_duration', 'genre_tags_musicbrainz',
    'first_year_on_chart_songs', 'genre_tags_through_first_top_10_hit',
    'major_genre_categories_through_first_top_10_hit',
    '#_of_major_genre_categories_through_first_top_10_hit',
    'musicbrainz_major_genre_categories', 'musicbrainz_#_of_genres',
    'spotify_genres', 'spotify_major_genre_categories',
]

df_artists_final = (
    df_artists[df_artists['first_top_20_hit_year'].between(2000, 2019)]
    .drop(columns=[c for c in cols_to_drop if c in df_artists.columns])
    .rename(columns={'combined_major_genre_categories': 'combined_major_genre_categories_artist'})
    .reset_index(drop=True)
)

print(df_artists_final.shape)
df_artists_final.head()


(778, 26)


,name,first_top_20_hit_year,first_charting_song_year,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit,top_20_hit_song_#_wks_on_chart_any_position,combined_major_genre_categories_artist,first_top_20_song_major_genres,first_top_20_song_duration_ms,first_top_20_song_acousticness,first_top_20_song_danceability,first_top_20_song_energy,first_top_20_song_instrumentalness,first_top_20_song_liveness,first_top_20_song_loudness,first_top_20_song_speechiness,first_top_20_song_tempo,first_top_20_song_valence,first_top_20_song_mode,first_top_20_song_explicit,degree_centrality_top20_rolling5,harmonic_closeness_centrality_top20_rolling5,betweenness_centrality_top20_rolling5,eigenvector_centrality_top20_rolling5,power_of_connected_artists_top20_rolling5,top_20_hitmaker
0,2 chainz,2012.0,2012.0,1.0,9.0,27.0,Hip Hop/Rap,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006764,5314.363492,0.010130,1.510211e-01,7104.5,1.0
1,21 savage,2017.0,2016.0,2.0,15.0,41.0,Hip Hop/Rap,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002871,5233.436039,0.001326,7.977090e-02,3116.5,1.0
2,3 doors down,2000.0,2000.0,1.0,2.0,53.0,"Metal, Pop, Rock","Blues, Metal, Rock",233933.0,0.00664,0.545,0.865,0.000011,0.168,-5.708,0.0286,99.009,0.543,0.0,0.0,NaN,NaN,NaN,NaN,NaN,1.0
3,3oh!3,2008.0,2008.0,1.0,1.0,37.0,"Electronic/Dance, Hip Hop/Rap, Pop, Punk/Hardc...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
4,5 seconds of summer,2014.0,2014.0,1.0,6.0,20.0,"Pop, Punk/Hardcore",Pop,237248.0,0.02830,0.572,0.499,0.000000,0.213,-5.237,0.0311,101.593,0.108,1.0,0.0,0.000048,1.000000,0.000000,1.555431e-18,NaN,1.0


#### *Don't run both of the next 2 cells. Choose one. The first cell one-hot-encodes the artist genre data and the second cell one-hot-encodes the song genre data. The former has fewer null values.*

In [3]:
# One-hot encode combined_major_genre_categories_artist into binary columns in df_artists_final.
# Splits comma-separated genre strings, creates one binary column per genre,
# prefixed with 'artist_genre_', replacing the original column.

artist_genre_dummies = (
    df_artists_final['combined_major_genre_categories_artist']
    .fillna('')
    .str.split(', ')
    .explode()
    .str.strip()
    .pipe(lambda s: pd.get_dummies(s, prefix='artist_genre'))
    .groupby(level=0)
    .max()
)

# Remove the empty-string column if it exists (from NaN rows)
artist_genre_dummies = artist_genre_dummies.loc[:, artist_genre_dummies.columns != 'artist_genre_']

# Insert genre columns where combined_major_genre_categories_artist was, then drop original
insert_at = df_artists_final.columns.get_loc('combined_major_genre_categories_artist')
df_artists_final = pd.concat([
    df_artists_final.iloc[:, :insert_at],
    artist_genre_dummies,
    df_artists_final.iloc[:, insert_at + 1:]
], axis=1)

# Add binary flag for artists with no genre data — inserted after last genre column
all_genre_cols = [c for c in df_artists_final.columns if c.startswith(('artist_genre_', 'song_genre_'))]
last_genre_idx = df_artists_final.columns.get_loc(all_genre_cols[-1])
df_artists_final.insert(last_genre_idx + 1, 'artist_genre_unknown', (df_artists_final[all_genre_cols].sum(axis=1) == 0).astype(int))

print("Artist genre columns added:", artist_genre_dummies.columns.tolist())
print("df_artists_final shape:", df_artists_final.shape)


Artist genre columns added: ['artist_genre_Blues', 'artist_genre_Classical', 'artist_genre_Country/Americana', 'artist_genre_Easy Listening/Vocal', 'artist_genre_Electronic/Dance', 'artist_genre_Experimental/Avant-Garde', 'artist_genre_Folk', 'artist_genre_Gospel/Christian/Religious', 'artist_genre_Hip Hop/Rap', 'artist_genre_Jazz', 'artist_genre_Latin', 'artist_genre_Metal', 'artist_genre_Pop', 'artist_genre_Punk/Hardcore', 'artist_genre_R&B/Soul/Funk', 'artist_genre_Reggae/Caribbean', 'artist_genre_Rock', 'artist_genre_World Music']
df_artists_final shape: (778, 44)


In [ ]:
# One-hot encode first_top_20_song_major_genres into binary columns in df_artists_final.
# Splits comma-separated genre strings, creates one binary column per genre,
# prefixed with 'song_genre_', replacing the original column.
# Also adds song_genre_unknown flag for artists with no song genre data.

song_genre_dummies = (
    df_artists_final['first_top_20_song_major_genres']
    .fillna('')
    .str.split(', ')
    .explode()
    .str.strip()
    .pipe(lambda s: pd.get_dummies(s, prefix='song_genre'))
    .groupby(level=0)
    .max()
)

# Remove the empty-string column if it exists (from NaN rows)
song_genre_dummies = song_genre_dummies.loc[:, song_genre_dummies.columns != 'song_genre_']

# Add song_genre_unknown flag to dummies before inserting
song_genre_dummies['song_genre_unknown'] = (song_genre_dummies.sum(axis=1) == 0).astype(int)

# Insert genre columns where first_top_20_song_major_genres was, then drop original
insert_at = df_artists_final.columns.get_loc('first_top_20_song_major_genres')
df_artists_final = pd.concat([
    df_artists_final.iloc[:, :insert_at],
    song_genre_dummies,
    df_artists_final.iloc[:, insert_at + 1:]
], axis=1)

print("Song genre columns added:", song_genre_dummies.columns.tolist())
print("df_artists_final shape:", df_artists_final.shape)


In [5]:
# Drop rows with null top_20_hitmaker (artists with no verified top-20 songs)
# and any duplicate rows, then confirm shape and class balance.

df_artists_final = (
    df_artists_final
    .dropna(subset=['top_20_hitmaker'])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(df_artists_final.shape)
print()
print(df_artists_final['top_20_hitmaker'].value_counts())


(759, 44)

top_20_hitmaker
0.0    431
1.0    328
Name: count, dtype: int64


In [6]:
# Drop name column, cast bool genre columns to int (0/1) in df_artists_final.

genre_cols = [c for c in df_artists_final.columns if c.startswith('genre_')]

df_artists_final = df_artists_final.drop(columns=['name'])
df_artists_final[genre_cols] = df_artists_final[genre_cols].astype(int)

print(df_artists_final.shape)
print(df_artists_final[genre_cols].dtypes.unique())


(759, 43)
[]


#### *Check this next cell for decisions regarding null values*

In [ ]:
# Impute missing values in df_artists_final:
# - Network metrics → fill with 0 (no collaboration data = no centrality)
# - Weeks on chart → fill with median
# - Spotify audio features → add has_spotify_data flag after first_top_20_song_mode, then fill with median

network_metric_cols = [
    'degree_centrality', 'harmonic_closeness', 'betweenness_centrality',
    'eigenvector_centrality', 'power'
]
for col in network_metric_cols:
    if col in df_artists_final.columns:
        df_artists_final[col] = df_artists_final[col].fillna(0)

wks_col = 'top_20_hit_song_#_wks_on_chart_any_position'
if wks_col in df_artists_final.columns:
    df_artists_final[wks_col] = df_artists_final[wks_col].fillna(
        df_artists_final[wks_col].median()
    )

exclude = {'first_top_20_song_explicit', 'first_top_20_song_major_genres'}
spotify_cols = [
    c for c in df_artists_final.columns
    if c.startswith('first_top_20_song_') and c not in exclude
]

for col in spotify_cols:
    df_artists_final[col] = df_artists_final[col].fillna(
        df_artists_final[col].median()
    )

# Insert has_spotify_data after first_top_20_song_mode
has_spotify = df_artists_final[spotify_cols].notna().any(axis=1).astype(int)
insert_at = df_artists_final.columns.get_loc('first_top_20_song_mode') + 1
df_artists_final.insert(insert_at, 'has_spotify_data', has_spotify)

# Drop explicit only if it still exists
if 'first_top_20_song_explicit' in df_artists_final.columns:
    df_artists_final = df_artists_final.drop(columns=['first_top_20_song_explicit'])

print("Imputation complete.")
print(df_artists_final.isnull().sum()[df_artists_final.isnull().sum() > 0])


Imputation complete.
first_top_20_song_major_genres                  391
degree_centrality_top20_rolling5                206
harmonic_closeness_centrality_top20_rolling5    206
betweenness_centrality_top20_rolling5           206
eigenvector_centrality_top20_rolling5           206
power_of_connected_artists_top20_rolling5       246
dtype: int64


In [10]:
# Fill null network metric columns with 0 (no collaboration data = no centrality)
network_cols = [
    'degree_centrality_top20_rolling5',
    'harmonic_closeness_centrality_top20_rolling5',
    'betweenness_centrality_top20_rolling5',
    'eigenvector_centrality_top20_rolling5',
    'power_of_connected_artists_top20_rolling5',
]
df_artists_final[network_cols] = df_artists_final[network_cols].fillna(0)
print(df_artists_final[network_cols].isnull().sum())


degree_centrality_top20_rolling5                0
harmonic_closeness_centrality_top20_rolling5    0
betweenness_centrality_top20_rolling5           0
eigenvector_centrality_top20_rolling5           0
power_of_connected_artists_top20_rolling5       0
dtype: int64


In [11]:
df_artists_final.head()

,first_top_20_hit_year,first_charting_song_year,years_through_first_top_20_hit,#_of_charting_songs_through_first_top_20_hit,top_20_hit_song_#_wks_on_chart_any_position,artist_genre_Blues,artist_genre_Classical,artist_genre_Country/Americana,artist_genre_Easy Listening/Vocal,artist_genre_Electronic/Dance,artist_genre_Experimental/Avant-Garde,artist_genre_Folk,artist_genre_Gospel/Christian/Religious,artist_genre_Hip Hop/Rap,artist_genre_Jazz,artist_genre_Latin,artist_genre_Metal,artist_genre_Pop,artist_genre_Punk/Hardcore,artist_genre_R&B/Soul/Funk,artist_genre_Reggae/Caribbean,artist_genre_Rock,artist_genre_World Music,artist_genre_unknown,first_top_20_song_major_genres,first_top_20_song_duration_ms,first_top_20_song_acousticness,first_top_20_song_danceability,first_top_20_song_energy,first_top_20_song_instrumentalness,first_top_20_song_liveness,first_top_20_song_loudness,first_top_20_song_speechiness,first_top_20_song_tempo,first_top_20_song_valence,first_top_20_song_mode,degree_centrality_top20_rolling5,harmonic_closeness_centrality_top20_rolling5,betweenness_centrality_top20_rolling5,eigenvector_centrality_top20_rolling5,power_of_connected_artists_top20_rolling5,top_20_hitmaker,has_spotify_data
0,2012.0,2012.0,1.0,9.0,27.0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,0,NaN,228000.0,0.06690,0.662,0.720,0.000000,0.121,-5.428,0.0556,119.987,0.527,1.0,0.006764,5314.363492,0.010130,1.510211e-01,7104.5,1.0,0
1,2017.0,2016.0,2.0,15.0,41.0,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,0,NaN,228000.0,0.06690,0.662,0.720,0.000000,0.121,-5.428,0.0556,119.987,0.527,1.0,0.002871,5233.436039,0.001326,7.977090e-02,3116.5,1.0,0
2,2000.0,2000.0,1.0,2.0,53.0,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,True,False,0,"Blues, Metal, Rock",233933.0,0.00664,0.545,0.865,0.000011,0.168,-5.708,0.0286,99.009,0.543,0.0,0.000000,0.000000,0.000000,0.000000e+00,0.0,1.0,1
3,2008.0,2008.0,1.0,1.0,37.0,False,False,False,False,True,False,False,False,True,False,False,False,True,True,False,False,True,False,0,NaN,228000.0,0.06690,0.662,0.720,0.000000,0.121,-5.428,0.0556,119.987,0.527,1.0,0.000000,0.000000,0.000000,0.000000e+00,0.0,1.0,0
4,2014.0,2014.0,1.0,6.0,20.0,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,False,0,Pop,237248.0,0.02830,0.572,0.499,0.000000,0.213,-5.237,0.0311,101.593,0.108,1.0,0.000048,1.000000,0.000000,1.555431e-18,0.0,1.0,1


## Model

In [12]:
# This will change based on whether you use artist genre info or song genre info

# Drop columns not used as model features
cols_to_drop = ['first_top_20_song_major_genres', 'first_top_20_hit_year', 'first_charting_song_year']
df_artists_final = df_artists_final.drop(columns=[c for c in cols_to_drop if c in df_artists_final.columns])
print(df_artists_final.shape)


(759, 40)


In [13]:
# Separate features and target
X = df_artists_final.drop(columns=['top_20_hitmaker'])
y = df_artists_final['top_20_hitmaker']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\ny value counts:\n{y.value_counts()}")
print(f"\ny class balance:\n{y.value_counts(normalize=True).round(3)}")


X shape: (759, 39)
y shape: (759,)

y value counts:
top_20_hitmaker
0.0    431
1.0    328
Name: count, dtype: int64

y class balance:
top_20_hitmaker
0.0    0.568
1.0    0.432
Name: proportion, dtype: float64


In [14]:
# Split into train and test sets, stratified to preserve class balance
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"\nTrain class balance:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest class balance:\n{y_test.value_counts(normalize=True).round(3)}")


Train: (607, 39), Test: (152, 39)

Train class balance:
top_20_hitmaker
0.0    0.568
1.0    0.432
Name: proportion, dtype: float64

Test class balance:
top_20_hitmaker
0.0    0.566
1.0    0.434
Name: proportion, dtype: float64


In [15]:
# Scale features for models sensitive to feature magnitude (Logistic Regression, SVM)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print(f"X_train_scaled shape: {X_train_scaled.shape}")


Scaling complete.
X_train_scaled shape: (607, 39)


In [19]:
# Evaluate models using 5-fold cross-validation on training set only.
# Test holdout is reserved for final evaluation after hyperparameter tuning.
# Logistic Regression and SVM use scaled features; tree-based models use unscaled.
# return_train_score=True enables overfitting check (train score vs CV score gap).

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate
from xgboost import XGBClassifier
import numpy as np

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

models = {
    'Dummy':               (DummyClassifier(strategy='stratified', random_state=42), X_train),
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), X_train_scaled),
    'XGBoost':             (XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'), X_train),
    'Random Forest':       (RandomForestClassifier(n_estimators=100, random_state=42), X_train),
    'SVM':                 (SVC(probability=True, random_state=42), X_train_scaled),
}

results = []
for name, (model, X_tr) in models.items():
    cv = cross_validate(model, X_tr, y_train, cv=5, scoring=scoring, return_train_score=True)
    results.append({
        'Model':             name,
        'Accuracy':          cv['test_accuracy'].mean(),
        'Precision':         cv['test_precision'].mean(),
        'Recall':            cv['test_recall'].mean(),
        'F1':                cv['test_f1'].mean(),
        'ROC-AUC (CV)':      cv['test_roc_auc'].mean(),
        'ROC-AUC (Train)':   cv['train_roc_auc'].mean(),
        'Overfit Gap':       cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean(),
    })

df_results = pd.DataFrame(results).set_index('Model').round(3)
print(df_results)


                     Accuracy  Precision  Recall     F1  ROC-AUC (CV)  \
Model                                                                   
Dummy                   0.503      0.421   0.405  0.413         0.491   
Logistic Regression     0.641      0.604   0.496  0.541         0.691   
XGBoost                 0.656      0.612   0.557  0.582         0.710   
Random Forest           0.659      0.627   0.519  0.564         0.737   
SVM                     0.652      0.637   0.461  0.532         0.719   

                     ROC-AUC (Train)  Overfit Gap  
Model                                              
Dummy                          0.490       -0.001  
Logistic Regression            0.778        0.087  
XGBoost                        0.998        0.288  
Random Forest                  0.989        0.252  
SVM                            0.903        0.184  
